In [1]:
# Correlation Analysis
# Examine correlations between renewable and non-renewable ETFs

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Define renewable vs non-renewable ETF groups
renewable_etfs = ['ICLN', 'PBW', 'QCLN']
nonrenewable_etfs = ['IXC', 'VDE', 'XLE']

# Calculate correlation matrix
corr_matrix = returns_pivot[renewable_etfs + nonrenewable_etfs].corr()

# Visualize correlation matrix with a heatmap
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", 
            mask=mask, vmin=-1, vmax=1, center=0,
            square=True, linewidths=.5)
plt.title('Correlation Matrix: Renewable vs Non-Renewable ETFs')
plt.tight_layout()
plt.show()

# Calculate average correlations between and within groups
def avg_correlation(corr_df, group1, group2=None):
    if group2 is None:  # Within-group correlation
        pairs = [(i, j) for i in range(len(group1)) for j in range(i+1, len(group1))]
        if not pairs:  # Handle single-element group
            return 0
        return sum(corr_df.loc[group1[i], group1[j]] for i, j in pairs) / len(pairs)
    else:  # Between-group correlation
        pairs = [(i, j) for i in range(len(group1)) for j in range(len(group2))]
        return sum(corr_df.loc[group1[i], group2[j]] for i, j in pairs) / len(pairs)

# Calculate average correlations for different market regimes
def regime_correlations(returns, etf_groups, sp500_col='SP500', window=60):
    # Identify bull/bear markets based on S&P 500 rolling returns
    bull_bear = returns[sp500_col].rolling(window).mean() > 0
    
    # Calculate correlations for different regimes
    results = {}
    for regime_name, regime_mask in [("Bull Market", bull_bear), ("Bear Market", ~bull_bear)]:
        if sum(regime_mask) > window:  # Ensure enough data points
            regime_returns = returns[regime_mask]
            regime_corr = regime_returns[etf_groups[0] + etf_groups[1]].corr()
            
            results[regime_name] = {
                "Within Renewable": avg_correlation(regime_corr, etf_groups[0]),
                "Within Non-Renewable": avg_correlation(regime_corr, etf_groups[1]),
                "Between Groups": avg_correlation(regime_corr, etf_groups[0], etf_groups[1])
            }
    
    return pd.DataFrame(results)

# Calculate and display regime correlations
regime_corrs = regime_correlations(returns_pivot, [renewable_etfs, nonrenewable_etfs])
print("\nCorrelation Analysis by Market Regime:")
print(regime_corrs)

# Visualize correlation changes during stress periods
stress_corrs = {}
for period_name, (start, end) in stress_periods.items():
    period_returns = returns_pivot.loc[start:end]
    period_corr = period_returns[renewable_etfs + nonrenewable_etfs].corr()
    stress_corrs[period_name] = {
        "Within Renewable": avg_correlation(period_corr, renewable_etfs),
        "Within Non-Renewable": avg_correlation(period_corr, nonrenewable_etfs),
        "Between Groups": avg_correlation(period_corr, renewable_etfs, nonrenewable_etfs)
    }

stress_corrs_df = pd.DataFrame(stress_corrs)
print("\nCorrelation Analysis during Stress Periods:")
print(stress_corrs_df)

# Visualize correlation changes during stress periods
stress_corrs_df.T.plot(kind='bar', figsize=(12, 6))
plt.title('Average Correlations During Stress Periods')
plt.ylabel('Correlation Coefficient')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title="Correlation Group")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

NameError: name 'returns_pivot' is not defined